# Análise do Sistema Financeiro e Crédito no Brasil (2015–2024)

**Projeto G2 · Tema 28** · Python · Pandas · Plotly · Streamlit · GitHub

**Sumário**

1. Introdução
2. Contextualização financeira
3. Explicação da base
4. Leitura dos dados
5. Limpeza e preparação
6. Engenharia de atributos
7. KPIs
8. Visualizações (com leitura de cada gráfico)
9. Interpretação dos resultados
10. Conclusão

## 1. Introdução

Este notebook investiga indicadores de **concessão de crédito, juros, inadimplência e risco** no sistema financeiro brasileiro entre **2015 e 2024**. Ele é a base analítica do dashboard em Streamlit (`app.py`): as funções de preparação e os gráficos aqui construídos serão reaproveitados no aplicativo.

**Perguntas orientadoras**

1. Quais regiões apresentam maior volume de crédito?
2. Quais setores possuem maior inadimplência?
3. Houve crescimento do crédito ao longo do tempo?
4. Existe relação entre juros e inadimplência?
5. Quais modalidades financeiras apresentam maior risco?
6. Como o comportamento financeiro evoluiu?
7. Quais estados possuem maior concentração bancária?

> **Aviso metodológico:** a base fornecida é **simulada**. Os resultados descrevem o comportamento *desta base* e não devem ser lidos como retrato do mercado de crédito real. Esse ponto é retomado na interpretação e na conclusão.

## 2. Contextualização financeira

O sistema financeiro canaliza poupança para consumo e investimento. Alguns conceitos usados ao longo da análise:

- **Concessão de crédito:** valor emprestado a pessoas e empresas. É a medida de "atividade" do sistema e reflete demanda, oferta e confiança.
- **Taxa de juros:** preço do crédito. Embute o custo de captação, a margem do banco e um prêmio pelo risco de não pagamento.
- **Inadimplência:** parcela da carteira com atraso no pagamento. É o principal indicador de risco *realizado*.
- **Risco de crédito (Baixo, Médio, Alto):** classificação *esperada* do tomador ou da operação.
- **Prazo médio de pagamento:** horizonte do empréstimo; prazos longos expõem o credor a mais incertezas.
- **Modalidades:** cartão, crédito pessoal, empresarial, imobiliário e veículos, que diferem em garantias, prazos e custo.

**Hipótese econômica a testar:** em mercados reais, juros mais altos costumam acompanhar maior risco percebido, e há debate sobre juros elevados também aumentarem a inadimplência (seleção adversa e risco moral). Na seção 8.5 testamos se essa relação aparece nos dados.

## 3. Explicação da base

Arquivo: `dados/simulacao_sistema_financeiro_brasil.csv`. Cada linha é um **registro mensal** de uma UF, para uma combinação de modalidade, risco e setor.

| Coluna | Tipo | Descrição |
|---|---|---|
| `ano`, `mes` | inteiro | Período de referência |
| `data` | data | Primeiro dia do mês de referência |
| `regiao`, `uf` | texto | Região e estado |
| `modalidade_credito` | texto | Cartão, Crédito pessoal, Empresarial, Imobiliário, Veículos |
| `valor_credito` | decimal | Valor concedido (R$) |
| `taxa_juros` | decimal | Taxa média de juros (%) |
| `inadimplencia_percentual` | decimal | Índice de inadimplência (%) |
| `quantidade_clientes` | inteiro | Total de clientes |
| `renda_media` | decimal | Renda média regional (R$) |
| `prazo_medio_pagamento` | decimal | Prazo médio (assumido em **dias**) |
| `risco_credito` | texto | Baixo, Médio ou Alto |
| `setor_economico` | texto | Agropecuária, Comércio, Indústria ou Serviços |

## 4. Leitura dos dados

In [ ]:
from pathlib import Path
import math

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
px.defaults.template = "plotly_white"

# Funciona tanto rodando de notebooks/ quanto da raiz do projeto
BASE = Path("..") if Path("../dados").exists() else Path(".")
CAMINHO_CSV = BASE / "dados" / "simulacao_sistema_financeiro_brasil.csv"
IMAGENS = BASE / "imagens"
IMAGENS.mkdir(exist_ok=True)

MESES = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun", "Jul", "Ago", "Set", "Out", "Nov", "Dez"]
ORDEM_RISCO = ["Baixo", "Médio", "Alto"]
COR_RISCO = {"Baixo": "#2e9e6b", "Médio": "#f0a500", "Alto": "#d64545"}


def fmt_num(v, casas=2):
    """Formata número no padrão brasileiro (1.234,56)."""
    return f"{v:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def fmt_moeda(v):
    if abs(v) >= 1e9:
        return f"R$ {fmt_num(v / 1e9)} bi"
    if abs(v) >= 1e6:
        return f"R$ {fmt_num(v / 1e6)} mi"
    return f"R$ {fmt_num(v)}"


def mostrar(fig, nome):
    """Exibe a figura e tenta salvá-la em imagens/ (exige kaleido; se ausente, só exibe)."""
    fig.show()
    try:
        fig.write_image(IMAGENS / f"{nome}.png", scale=2)
    except Exception:
        pass

In [ ]:
# utf-8-sig remove o BOM que vem no início do arquivo (evita a coluna "\ufeffano")
df_raw = pd.read_csv(CAMINHO_CSV, encoding="utf-8-sig")
print(f"Linhas: {df_raw.shape[0]:,} | Colunas: {df_raw.shape[1]}")
display(df_raw.head())
df_raw.info()

## 5. Limpeza e preparação

Etapas: ajuste de tipos, nulos, duplicatas, coerência entre colunas, regras de domínio e outliers.

In [ ]:
df = df_raw.copy()

# 5.1 Tipos e padronização de texto
df["data"] = pd.to_datetime(df["data"], errors="coerce")
colunas_texto = ["regiao", "uf", "modalidade_credito", "risco_credito", "setor_economico"]
for c in colunas_texto:
    df[c] = df[c].str.strip()

# 5.2 Nulos e duplicatas
nulos = df.isna().sum()
print("Valores nulos:", "nenhum" if not nulos.any() else nulos[nulos > 0].to_dict())
print("Linhas duplicadas:", df.duplicated().sum())

# 5.3 Coerência entre ano/mês e data
divergentes = ~((df["data"].dt.year == df["ano"]) & (df["data"].dt.month == df["mes"]))
print("Registros com ano/mês diferentes da data:", divergentes.sum())

# 5.4 Coerência UF x região (mapa oficial do IBGE)
UF_REGIAO = {
    **dict.fromkeys(["AC", "AM", "AP", "PA", "RO", "RR", "TO"], "Norte"),
    **dict.fromkeys(["AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"], "Nordeste"),
    **dict.fromkeys(["DF", "GO", "MS", "MT"], "Centro-Oeste"),
    **dict.fromkeys(["ES", "MG", "RJ", "SP"], "Sudeste"),
    **dict.fromkeys(["PR", "RS", "SC"], "Sul"),
}
uf_incoerente = df["uf"].map(UF_REGIAO) != df["regiao"]
print("Registros com UF fora da região informada:", uf_incoerente.sum())

# 5.5 Regras de domínio
regras = {
    "valor_credito > 0": df["valor_credito"] > 0,
    "taxa_juros entre 0 e 100": df["taxa_juros"].between(0, 100),
    "inadimplencia entre 0 e 100": df["inadimplencia_percentual"].between(0, 100),
    "quantidade_clientes > 0": df["quantidade_clientes"] > 0,
    "renda_media > 0": df["renda_media"] > 0,
    "prazo_medio_pagamento > 0": df["prazo_medio_pagamento"] > 0,
}
print("\nViolações de regras de domínio:")
display(pd.Series({k: int((~v).sum()) for k, v in regras.items()}, name="violações"))

In [ ]:
# 5.6 Outliers (critério IQR) e estatísticas descritivas
colunas_num = ["valor_credito", "taxa_juros", "inadimplencia_percentual",
               "quantidade_clientes", "renda_media", "prazo_medio_pagamento"]


def contar_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum())


diagnostico = df[colunas_num].describe().T
diagnostico["outliers_IQR"] = df[colunas_num].apply(contar_outliers)
display(diagnostico)

print("Máximo de registros por UF/mês:", df.groupby(["ano", "mes", "uf"]).size().max())
print("Registros por mês (mín–máx):", df.groupby("data").size().min(), "–", df.groupby("data").size().max())

In [ ]:
# 5.7 Formato das distribuições
fig = make_subplots(rows=1, cols=3, subplot_titles=("Taxa de juros (%)", "Inadimplência (%)", "Valor do crédito (R$)"))
fig.add_trace(go.Histogram(x=df["taxa_juros"], nbinsx=30, marker_color="#1f4e79", showlegend=False), row=1, col=1)
fig.add_trace(go.Histogram(x=df["inadimplencia_percentual"], nbinsx=30, marker_color="#d64545", showlegend=False), row=1, col=2)
fig.add_trace(go.Histogram(x=df["valor_credito"], nbinsx=30, marker_color="#2e9e6b", showlegend=False), row=1, col=3)
fig.update_layout(title="Distribuição das principais variáveis numéricas", height=380)
mostrar(fig, "00_distribuicoes")

**Diagnóstico da base**

- A base está **íntegra**: sem nulos, duplicatas, datas incoerentes, UFs fora da região ou valores fora dos limites. Os únicos ajustes necessários foram de **tipo** (`data`) e de **codificação** (BOM).
- Os histogramas são **aproximadamente planos (distribuição uniforme)**: juros entre ~6% e 90%, inadimplência entre 1% e 20%, valores entre ~R$ 0,1 mi e R$ 10 mi. Dados reais costumam ser assimétricos, com poucos valores extremos. Isso indica geração aleatória e antecipa que **não haverá correlações fortes** entre as variáveis.
- Sem outliers pelo critério IQR, **nenhum registro foi removido**.
- A base cobre **20 das 27 UFs**, o que limita comparações "nacionais".

## 6. Engenharia de atributos

Variáveis derivadas para facilitar a análise temporal, a segmentação e a leitura de risco.

In [ ]:
# Tempo
df["trimestre"] = df["data"].dt.quarter
df["semestre"] = np.where(df["mes"] <= 6, 1, 2)
df["ano_trimestre"] = df["ano"].astype(str) + "-T" + df["trimestre"].astype(str)
df["mes_nome"] = df["mes"].map(dict(enumerate(MESES, start=1)))

# Intensidade do crédito e exposição ao risco
df["valor_por_cliente"] = df["valor_credito"] / df["quantidade_clientes"]
df["valor_inadimplente_est"] = df["valor_credito"] * df["inadimplencia_percentual"] / 100

# Faixas para segmentação
df["faixa_juros"] = pd.cut(df["taxa_juros"], bins=[0, 20, 40, 60, 80, 100],
                           labels=["0–20%", "20–40%", "40–60%", "60–80%", "80–100%"], include_lowest=True)
df["faixa_inadimplencia"] = pd.cut(df["inadimplencia_percentual"], bins=[0, 5, 10, 15, 20],
                                   labels=["Baixa (≤5%)", "Moderada (5–10%)", "Alta (10–15%)", "Crítica (>15%)"],
                                   include_lowest=True)
df["faixa_prazo"] = pd.cut(df["prazo_medio_pagamento"], bins=[0, 60, 120, 180, 240],
                           labels=["até 60 dias", "61–120", "121–180", "181–240"], include_lowest=True)

# Risco como categoria ordenada + versão numérica (1 = Baixo, 3 = Alto)
df["risco_credito"] = pd.Categorical(df["risco_credito"], categories=ORDEM_RISCO, ordered=True)
df["risco_num"] = df["risco_credito"].map({"Baixo": 1, "Médio": 2, "Alto": 3}).astype(int)

novas = ["trimestre", "semestre", "ano_trimestre", "mes_nome", "valor_por_cliente",
         "valor_inadimplente_est", "faixa_juros", "faixa_inadimplencia", "faixa_prazo", "risco_num"]
display(df[["data", "uf", "valor_credito", "taxa_juros", "inadimplencia_percentual"] + novas].head())

In [ ]:
def resumo(d, por):
    """Resumo por grupo. As taxas ponderadas usam o valor de crédito como peso,
    pois uma taxa de uma operação grande deve pesar mais que a de uma pequena."""
    t = d.assign(_juros_w=d["taxa_juros"] * d["valor_credito"],
                 _inad_w=d["inadimplencia_percentual"] * d["valor_credito"])
    out = t.groupby(por, observed=True).agg(
        operacoes=("valor_credito", "size"),
        volume=("valor_credito", "sum"),
        credito_medio=("valor_credito", "mean"),
        clientes=("quantidade_clientes", "sum"),
        juros_medio=("taxa_juros", "mean"),
        inad_media=("inadimplencia_percentual", "mean"),
        prazo_medio=("prazo_medio_pagamento", "mean"),
        _juros_w=("_juros_w", "sum"),
        _inad_w=("_inad_w", "sum"),
    )
    out["juros_pond"] = out["_juros_w"] / out["volume"]
    out["inad_pond"] = out["_inad_w"] / out["volume"]
    return out.drop(columns=["_juros_w", "_inad_w"])

## 7. KPIs

Os seis KPIs exigidos. Para taxa de juros e inadimplência mostramos a média **simples** e a **ponderada pelo valor de crédito**. A "modalidade mais utilizada" depende do critério, então mostramos três.

In [ ]:
vol_total = df["valor_credito"].sum()
juros_simples = df["taxa_juros"].mean()
juros_pond = np.average(df["taxa_juros"], weights=df["valor_credito"])
inad_simples = df["inadimplencia_percentual"].mean()
inad_pond = np.average(df["inadimplencia_percentual"], weights=df["valor_credito"])
prazo_medio = df["prazo_medio_pagamento"].mean()

mod = resumo(df, "modalidade_credito")
reg = resumo(df, "regiao")

mod_volume = mod["volume"].idxmax()
mod_operacoes = mod["operacoes"].idxmax()
mod_clientes = mod["clientes"].idxmax()
reg_top = reg["volume"].idxmax()

kpis = pd.DataFrame([
    ("Volume total de crédito", fmt_moeda(vol_total), "Soma de valor_credito, 2015–2024"),
    ("Taxa média de juros", f"{fmt_num(juros_simples)}% (pond.: {fmt_num(juros_pond)}%)", "Média simples e ponderada pelo valor"),
    ("Índice médio de inadimplência", f"{fmt_num(inad_simples)}% (pond.: {fmt_num(inad_pond)}%)", "Média simples e ponderada pelo valor"),
    ("Modalidade mais utilizada", mod_volume, f"Por volume. Por operações: {mod_operacoes}; por clientes: {mod_clientes}"),
    ("Região com maior crédito", reg_top, f"{fmt_moeda(reg['volume'].max())} ({reg['volume'].max() / vol_total:.1%} do total)"),
    ("Prazo médio de pagamento", f"{fmt_num(prazo_medio, 1)} dias", "Média simples (unidade assumida: dias)"),
], columns=["KPI", "Valor", "Observação"])
display(kpis)

top2 = mod["volume"].nlargest(2)
print(f"\nMargem entre as duas primeiras modalidades por volume ({top2.index[0]} x {top2.index[1]}): "
      f"{(top2.iloc[0] - top2.iloc[1]) / top2.iloc[0]:.1%}")

**Leitura dos KPIs**

- A **taxa média de juros (~48,7%)** e a **inadimplência média (~10,6%)** são consistentes entre média simples e ponderada. A ponderação quase não altera os valores porque as variáveis são independentes do valor da operação.
- A **modalidade líder muda conforme o critério** (volume, operações ou clientes), e a **margem entre as primeiras colocadas é pequena**. Dizer que uma modalidade é "a mais utilizada" exige citar o critério.
- A **região líder em crédito é o Sudeste**, mas a leitura regional precisa considerar o número de registros (seção 8.3).

## 8. Visualizações

Cada gráfico é seguido de uma leitura. As imagens são salvas em `imagens/` quando o `kaleido` está instalado.

### 8.1 Evolução temporal do crédito

In [ ]:
mensal = df.groupby("data", as_index=False).agg(volume=("valor_credito", "sum"))
mensal["volume_mi"] = mensal["volume"] / 1e6
mensal["media_movel_12m"] = mensal["volume_mi"].rolling(12).mean()

fig = go.Figure()
fig.add_trace(go.Scatter(x=mensal["data"], y=mensal["volume_mi"], mode="lines",
                         name="Volume mensal", line=dict(width=1.5, color="#9bb7d4")))
fig.add_trace(go.Scatter(x=mensal["data"], y=mensal["media_movel_12m"], mode="lines",
                         name="Média móvel de 12 meses", line=dict(width=3, color="#1f4e79")))
fig.update_layout(title="Evolução mensal do volume de crédito concedido",
                  yaxis_title="R$ milhões", xaxis_title="", hovermode="x unified")
mostrar(fig, "01_linha_temporal_credito")

anual = df.groupby("ano", as_index=False).agg(volume=("valor_credito", "sum"))
anual["volume_bi"] = anual["volume"] / 1e9
anual["variacao_pct"] = anual["volume"].pct_change() * 100
rotulos = anual["variacao_pct"].map(lambda v: "" if pd.isna(v) else f"{v:+.1f}%")

fig = px.bar(anual, x="ano", y="volume_bi", text=rotulos,
             labels={"volume_bi": "R$ bilhões", "ano": ""},
             title="Volume anual de crédito (rótulo = variação sobre o ano anterior)")
fig.update_xaxes(type="category")
fig.update_traces(textposition="outside", marker_color="#1f4e79")
mostrar(fig, "02_volume_anual")
display(anual)

**Leitura.** O volume anual fica em torno de **R$ 2,2 bi** durante toda a série. De 2015 a 2024 o crescimento total é de apenas **~2,5%** (≈0,3% ao ano), ou seja, **estabilidade, não expansão**. As variações anuais são pequenas e sem direção clara: **2021 (+4,7%)** e **2023 (+3,8%)** sobem, **2020 (−2,8%)** e **2022 (−6,2%)** caem. A queda de 2020 poderia sugerir um efeito da pandemia, mas a de 2022 é maior e sem explicação macroeconômica óbvia, o que reforça que são oscilações aleatórias da simulação.

### 8.2 Comparação entre modalidades

In [ ]:
mod = resumo(df, "modalidade_credito").sort_values("volume")

fig = px.bar(mod, x=mod["volume"] / 1e9, y=mod.index, orientation="h", text=(mod["volume"] / 1e9).round(2),
             labels={"x": "R$ bilhões", "y": ""}, title="Volume de crédito por modalidade")
fig.update_traces(marker_color="#1f4e79", textposition="outside")
mostrar(fig, "03_volume_por_modalidade")

risco_mod = mod[["inad_pond", "juros_pond"]].reset_index().melt(
    id_vars="modalidade_credito", var_name="indicador", value_name="valor")
risco_mod["indicador"] = risco_mod["indicador"].map({"inad_pond": "Inadimplência ponderada (%)",
                                                      "juros_pond": "Juros ponderados (%)"})
fig = px.bar(risco_mod, x="modalidade_credito", y="valor", color="indicador", barmode="group",
             labels={"valor": "%", "modalidade_credito": ""}, title="Inadimplência e juros por modalidade (eixo a partir de zero)")
mostrar(fig, "04_risco_por_modalidade")

alto = (df["risco_credito"] == "Alto").groupby(df["modalidade_credito"]).mean().mul(100).round(1).rename("% operações de risco Alto")
display(mod.sort_values("volume", ascending=False)[["operacoes", "volume", "juros_pond", "inad_pond"]].join(alto))

**Leitura.** Os volumes ficam numa faixa estreita (**R$ 4,4 a 4,6 bi**), e **Cartão** lidera com margem de menos de 1% sobre **Empresarial**. Em risco, a inadimplência ponderada vai de ~10,4% (Veículos) a ~10,7% (Imobiliário), uma **diferença de ~0,2 p.p.**, pequena demais para classificar alguma modalidade como nitidamente mais arriscada. **Cartão** tem os juros mais altos (~50%), mas sem inadimplência maior. **Veículos** tem a maior proporção de operações classificadas como risco Alto (35%) e, ao mesmo tempo, a menor inadimplência, outro sinal de que a classificação de risco da base não acompanha o risco observado.

### 8.3 Comparação regional

In [ ]:
reg = resumo(df, "regiao")
ufs_por_regiao = df.groupby("regiao")["uf"].nunique()
reg["ufs"] = ufs_por_regiao
reg["volume_por_uf"] = reg["volume"] / reg["ufs"]
reg = reg.sort_values("volume", ascending=False)

fig = px.bar(reg, x=reg.index, y=reg["volume"] / 1e9, text=(reg["volume"] / 1e9).round(2),
             labels={"y": "R$ bilhões", "x": ""}, title="Volume total de crédito por região")
fig.update_traces(marker_color="#1f4e79", textposition="outside")
mostrar(fig, "05_volume_por_regiao")

comp = reg[["credito_medio", "volume_por_uf"]].div(1e6).reset_index().melt(
    id_vars="regiao", var_name="métrica", value_name="R$ milhões")
comp["métrica"] = comp["métrica"].map({"credito_medio": "Crédito médio por operação",
                                       "volume_por_uf": "Volume médio por UF"})
fig = px.bar(comp, x="regiao", y="R$ milhões", color="métrica", barmode="group", facet_col="métrica",
             title="Comparação regional normalizada (por operação e por UF)")
fig.update_yaxes(matches=None)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
mostrar(fig, "06_regiao_normalizada")

fig = px.bar(df.groupby(["regiao", "modalidade_credito"], as_index=False)["valor_credito"].sum(),
             x="regiao", y="valor_credito", color="modalidade_credito",
             labels={"valor_credito": "R$", "regiao": ""}, title="Composição do crédito por região e modalidade",
             category_orders={"regiao": list(reg.index)})
mostrar(fig, "07_regiao_modalidade")
display(reg[["operacoes", "ufs", "volume", "credito_medio", "volume_por_uf", "inad_pond", "juros_pond"]])

**Leitura.** O **Sudeste** lidera com **R$ 8,1 bi (~36% do total)**, seguido por Nordeste (R$ 4,8 bi). Mas a diferença **vem principalmente da quantidade de registros**: o Sudeste tem 1.560 operações contra 600 no Norte e no Centro-Oeste. Quando se compara o **crédito médio por operação**, as regiões ficam praticamente iguais (**~R$ 4,9 a 5,2 mi**). Portanto, o Sudeste lidera em *volume*, não em *intensidade*. O volume por UF também é afetado pelo número de registros de cada UF (por exemplo, RJ tem 480 linhas e outras UFs só 120).

### 8.4 Heatmap mensal

In [ ]:
pivot_vol = df.pivot_table(index="ano", columns="mes", values="valor_credito", aggfunc="sum") / 1e6
fig = px.imshow(pivot_vol, x=MESES, aspect="auto", color_continuous_scale="Blues", text_auto=".0f",
                labels=dict(x="Mês", y="Ano", color="R$ mi"), title="Heatmap: volume de crédito por ano e mês (R$ milhões)")
fig.update_yaxes(type="category")
mostrar(fig, "08_heatmap_volume")

pivot_inad = df.pivot_table(index="ano", columns="mes", values="inadimplencia_percentual", aggfunc="mean")
fig = px.imshow(pivot_inad, x=MESES, aspect="auto", color_continuous_scale="Reds", text_auto=".1f",
                labels=dict(x="Mês", y="Ano", color="%"), title="Heatmap: inadimplência média por ano e mês (%)")
fig.update_yaxes(type="category")
mostrar(fig, "09_heatmap_inadimplencia")

total_por_mes = df.groupby("mes")["valor_credito"].sum()
print(f"Mês de maior volume acumulado: {MESES[total_por_mes.idxmax() - 1]} ({fmt_moeda(total_por_mes.max())})")
print(f"Mês de menor volume acumulado: {MESES[total_por_mes.idxmin() - 1]} ({fmt_moeda(total_por_mes.min())})")

**Leitura.** O heatmap **não mostra um padrão sazonal estável**: as células mais claras e mais escuras mudam de posição de um ano para outro. No acumulado, **novembro** tem o maior volume e **setembro** o menor, mas essa diferença (~11%) precisaria se repetir ano após ano para ser chamada de sazonalidade, e aqui isso não ocorre de forma visível. O mesmo vale para a inadimplência, que flutua em torno de 10–11% sem trajetória definida.

### 8.5 Juros × inadimplência

In [ ]:
x, y = df["taxa_juros"], df["inadimplencia_percentual"]
r_pearson = x.corr(y)
r_spearman = x.corr(y, method="spearman")
n = len(df)
t_stat = r_pearson * math.sqrt((n - 2) / (1 - r_pearson ** 2))
p_valor = math.erfc(abs(t_stat) / math.sqrt(2))  # aproximação normal (n grande)

inclin, interc = np.polyfit(x, y, 1)
xs = np.linspace(x.min(), x.max(), 50)

fig = px.scatter(df, x="taxa_juros", y="inadimplencia_percentual", color="risco_credito", opacity=0.35,
                 color_discrete_map=COR_RISCO, category_orders={"risco_credito": ORDEM_RISCO},
                 labels={"taxa_juros": "Taxa de juros (%)", "inadimplencia_percentual": "Inadimplência (%)", "risco_credito": "Risco"},
                 title=f"Juros x inadimplência (Pearson r = {r_pearson:.3f}; p ≈ {p_valor:.2f})")
fig.add_trace(go.Scatter(x=xs, y=inclin * xs + interc, mode="lines", name="Tendência linear",
                         line=dict(color="black", width=3)))
mostrar(fig, "10_dispersao_juros_inadimplencia")

por_faixa = df.groupby("faixa_juros", observed=True)["inadimplencia_percentual"].agg(["mean", "size"]).reset_index()
fig = px.bar(por_faixa, x="faixa_juros", y="mean", text=por_faixa["mean"].round(2),
             labels={"mean": "Inadimplência média (%)", "faixa_juros": "Faixa de juros"},
             title="Inadimplência média por faixa de juros")
fig.update_traces(marker_color="#d64545", textposition="outside")
mostrar(fig, "11_inadimplencia_por_faixa_juros")

corr = df[colunas_num + ["risco_num"]].corr()
fig = px.imshow(corr, text_auto=".2f", zmin=-1, zmax=1, color_continuous_scale="RdBu_r",
                title="Matriz de correlação (Pearson)")
mostrar(fig, "12_matriz_correlacao")

print(f"Pearson = {r_pearson:.3f} | Spearman = {r_spearman:.3f} | p-valor ≈ {p_valor:.2f}")
print("Correlação juros x inadimplência por região:")
display(df.groupby("regiao")[["taxa_juros", "inadimplencia_percentual"]].corr().iloc[0::2, 1].droplevel(1).rename("pearson"))

**Leitura.** A correlação é **praticamente nula (r ≈ −0,01; p ≈ 0,59)**: não há evidência de relação entre juros e inadimplência. A nuvem de pontos é uniforme, a reta de tendência é quase horizontal, e a inadimplência média fica em ~10,4%–10,8% em **todas** as faixas de juros. Dentro de cada região os coeficientes também ficam muito próximos de zero (entre cerca de −0,04 e +0,04). A matriz de correlação mostra o mesmo para todas as variáveis.

Em dados reais, esperaríamos alguma relação positiva. A ausência aqui é **característica da simulação**, não uma descoberta econômica.

### 8.6 Tabela dinâmica

In [ ]:
pivot_regiao_mod = df.pivot_table(index="regiao", columns="modalidade_credito", values="valor_credito",
                                  aggfunc="sum", margins=True, margins_name="Total") / 1e6
print("Volume de crédito (R$ milhões) por região x modalidade")
display(pivot_regiao_mod.round(1))

pivot_setor_risco = df.pivot_table(index="setor_economico", columns="risco_credito", values="inadimplencia_percentual",
                                   aggfunc="mean", observed=True, margins=True, margins_name="Média")
print("\nInadimplência média (%) por setor x classificação de risco")
display(pivot_setor_risco.round(2))

**Leitura.** O Sudeste lidera em todas as modalidades, com destaque para **Cartão** e **Empresarial** (~R$ 1,7 bi cada). Na segunda tabela, a inadimplência **cresce de Baixo para Alto** em todos os setores, mas de forma muito leve (média de ~10,5% para ~10,8%, ou **~0,3 p.p.**). A classificação de risco da base, portanto, **discrimina muito pouco o risco realizado**.

### 8.7 Evolução do risco de crédito

In [ ]:
risco_ano = (pd.crosstab(df["ano"], df["risco_credito"], normalize="index") * 100).reset_index().melt(
    id_vars="ano", var_name="risco", value_name="%")
fig = px.bar(risco_ano, x="ano", y="%", color="risco", color_discrete_map=COR_RISCO,
             category_orders={"risco": ORDEM_RISCO}, title="Distribuição das operações por classificação de risco, por ano")
fig.update_xaxes(type="category")
mostrar(fig, "13_evolucao_risco")

risco_media = df.groupby("ano", as_index=False).agg(risco_medio=("risco_num", "mean"),
                                                     inadimplencia=("inadimplencia_percentual", "mean"))
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=risco_media["ano"], y=risco_media["risco_medio"], name="Risco médio (1–3)",
                         mode="lines+markers", line=dict(color="#f0a500", width=3)), secondary_y=False)
fig.add_trace(go.Scatter(x=risco_media["ano"], y=risco_media["inadimplencia"], name="Inadimplência média (%)",
                         mode="lines+markers", line=dict(color="#d64545", width=3)), secondary_y=True)
fig.update_layout(title="Risco médio e inadimplência média por ano")
fig.update_xaxes(type="category")
mostrar(fig, "14_risco_e_inadimplencia_ano")

ranking_risco = df.groupby("risco_credito", observed=True).agg(
    operacoes=("valor_credito", "size"), inadimplencia=("inadimplencia_percentual", "mean"), juros=("taxa_juros", "mean"))
display(ranking_risco)

**Leitura.** As três classes de risco ficam sempre em torno de **um terço** das operações, sem tendência de piora ou melhora. Em 2023, por exemplo, "Alto" teve só 121 operações, e em 2024 voltou a 158, uma oscilação sem direção. A inadimplência média anual varia entre **~10,2% e ~11,0%**. Além disso, operações classificadas como **Alto** têm inadimplência (~10,8%) e juros (~48,7%) quase iguais aos de **Baixo** (~10,5% e ~48,4%). Há um sinal na direção esperada (Alto > Baixo), mas de ~0,3 p.p. Em um mercado real, essa diferença seria bem maior.

### 8.8 Setor econômico

In [ ]:
setor = resumo(df, "setor_economico").sort_values("inad_pond", ascending=False)
fig = px.bar(setor, x=setor.index, y="inad_pond", text=setor["inad_pond"].round(2),
             labels={"inad_pond": "Inadimplência ponderada (%)", "setor_economico": ""},
             title="Inadimplência por setor econômico")
fig.update_traces(marker_color="#d64545", textposition="outside")
mostrar(fig, "15_inadimplencia_por_setor")

fig = px.bar(setor, x=setor.index, y=setor["volume"] / 1e9, text=(setor["volume"] / 1e9).round(2),
             labels={"y": "R$ bilhões", "setor_economico": ""}, title="Volume de crédito por setor econômico")
fig.update_traces(marker_color="#1f4e79", textposition="outside")
mostrar(fig, "16_volume_por_setor")
display(setor[["operacoes", "volume", "juros_pond", "inad_pond"]])

**Leitura.** **Comércio** tem a maior inadimplência ponderada (~10,9%) e **Serviços** a menor (~10,5%), uma diferença de **~0,4 p.p.** O volume de crédito é parecido entre os setores (R$ 5,4 a 5,7 bi). A diferença existe, mas é pequena e não há como saber, nesta base, se é estatisticamente relevante ou só variação aleatória.

### 8.9 Regiões críticas e concentração bancária

**Índice de criticidade por UF:** média dos ranks percentuais de três indicadores de risco (inadimplência ponderada, % de operações de risco Alto e juros ponderados). Quanto mais perto de 100, mais a UF aparece entre as piores.

In [ ]:
uf = resumo(df, "uf")
uf["regiao"] = df.groupby("uf")["regiao"].first()
uf["pct_alto_risco"] = (df["risco_credito"] == "Alto").groupby(df["uf"]).mean().mul(100)
uf["indice_criticidade"] = uf[["inad_pond", "pct_alto_risco", "juros_pond"]].rank(pct=True).mean(axis=1) * 100
uf["participacao_pct"] = uf["volume"] / uf["volume"].sum() * 100

critico = uf.sort_values("indice_criticidade", ascending=False).head(10)
fig = px.bar(critico, x=critico.index, y="indice_criticidade", color="regiao",
             text=critico["indice_criticidade"].round(0),
             labels={"indice_criticidade": "Índice de criticidade (0–100)", "uf": "UF"},
             title="Top 10 UFs mais críticas (inadimplência, risco Alto e juros)")
fig.update_traces(textposition="outside")
mostrar(fig, "17_ufs_criticas")

fig = px.scatter(uf, x=uf["volume"] / 1e9, y="inad_pond", size="clientes", color="regiao", text=uf.index,
                 labels={"x": "Volume de crédito (R$ bi)", "inad_pond": "Inadimplência ponderada (%)"},
                 title="Mapa de risco: volume x inadimplência por UF (tamanho = clientes)")
fig.update_traces(textposition="top center")
mostrar(fig, "18_mapa_risco_uf")

conc = uf.sort_values("participacao_pct", ascending=False)
conc["acumulado_pct"] = conc["participacao_pct"].cumsum()
fig = px.bar(conc, x=conc.index, y="participacao_pct", color="regiao", text=conc["participacao_pct"].round(1),
             labels={"participacao_pct": "% do volume total", "uf": "UF"},
             title="Concentração do crédito por UF (participação no volume total)")
fig.update_traces(textposition="outside")
mostrar(fig, "19_concentracao_uf")

display(conc[["regiao", "operacoes", "volume", "participacao_pct", "acumulado_pct", "inad_pond", "pct_alto_risco", "indice_criticidade"]])

**Leitura.**

- **Concentração:** as quatro maiores UFs (**RJ, ES, MG e SP**) são todas do Sudeste e somam **~36%** do volume. O **RJ** lidera isoladamente (~11%). Esse resultado reflete o fato de o Sudeste ter mais registros, e não uma intensidade maior de crédito por operação.
- **Regiões críticas:** no topo do ranking aparecem **PR, MS, ES, GO, AM e CE**. O índice combina mais inadimplência, mais operações de risco Alto e juros mais altos. Como as diferenças entre UFs são pequenas (inadimplência ponderada entre ~9,8% e ~11,4%) e muitas UFs têm só 120 registros, o ranking é **indicativo**: serve para orientar a investigação, não para concluir que uma UF é de fato mais arriscada.

## 9. Interpretação dos resultados

Respostas objetivas às perguntas orientadoras, calculadas diretamente da base.

In [ ]:
reg = resumo(df, "regiao")
mod = resumo(df, "modalidade_credito")
setor = resumo(df, "setor_economico")
uf_vol = df.groupby("uf")["valor_credito"].sum().sort_values(ascending=False)
anual = df.groupby("ano")["valor_credito"].sum()
inad_ano = df.groupby("ano")["inadimplencia_percentual"].mean()
cagr = (anual.iloc[-1] / anual.iloc[0]) ** (1 / (len(anual) - 1)) - 1

print("1) Região com maior volume:", reg["volume"].idxmax(),
      f"({fmt_moeda(reg['volume'].max())}, {reg['volume'].max() / reg['volume'].sum():.1%} do total).")
print(f"   Crédito médio por operação varia só de {fmt_moeda(reg['credito_medio'].min())} a {fmt_moeda(reg['credito_medio'].max())}.")

print("\n2) Setor com maior inadimplência:", setor["inad_pond"].idxmax(), f"({setor['inad_pond'].max():.2f}%)",
      "| menor:", setor["inad_pond"].idxmin(), f"({setor['inad_pond'].min():.2f}%)",
      f"| diferença: {setor['inad_pond'].max() - setor['inad_pond'].min():.2f} p.p.")

print(f"\n3) Crescimento 2015→2024: {anual.iloc[-1] / anual.iloc[0] - 1:+.1%} no total ({cagr:+.2%} ao ano).",
      f"Maior alta: {anual.pct_change().idxmax()} ({anual.pct_change().max():+.1%});",
      f"maior queda: {anual.pct_change().idxmin()} ({anual.pct_change().min():+.1%}).")

forca = "praticamente nula" if abs(r_pearson) < 0.1 else "fraca" if abs(r_pearson) < 0.3 else "moderada ou forte"
print(f"\n4) Correlação juros x inadimplência: r = {r_pearson:.3f} (Spearman {r_spearman:.3f}), p ≈ {p_valor:.2f} → relação {forca}.")

print("\n5) Modalidade com maior inadimplência:", mod["inad_pond"].idxmax(), f"({mod['inad_pond'].max():.2f}%)",
      "| menor:", mod["inad_pond"].idxmin(), f"({mod['inad_pond'].min():.2f}%).")

print(f"\n6) Inadimplência média anual: mínimo de {inad_ano.min():.2f}% ({inad_ano.idxmin()}) e máximo de {inad_ano.max():.2f}% ({inad_ano.idxmax()}).")

print("\n7) UF com maior participação no crédito:", uf_vol.index[0], f"({uf_vol.iloc[0] / uf_vol.sum():.1%}).",
      f"As 4 maiores UFs ({', '.join(uf_vol.index[:4])}) somam {uf_vol.iloc[:4].sum() / uf_vol.sum():.1%}.")

**Síntese interpretativa**

| Pergunta | Resposta | Confiança |
|---|---|---|
| Região com maior crédito | Sudeste, por quantidade de registros, não por crédito médio por operação | Alta no ranking, baixa como evidência de "maior intensidade" |
| Setor mais inadimplente | Comércio, com diferença de ~0,4 p.p. para Serviços | Baixa |
| Crescimento do crédito | Estabilidade (~+2,5% em 10 anos) | Alta |
| Juros × inadimplência | Sem relação (r ≈ −0,01) | Alta |
| Modalidade de maior risco | Imobiliário, mas por margem mínima | Baixa |
| Evolução do comportamento | Sem tendência nem sazonalidade claras | Média |
| Concentração por UF | RJ lidera; as 4 UFs do Sudeste somam ~36% | Alta no ranking, mas reflete o número de registros |

**Ressalva geral.** Quase todas as diferenças encontradas são pequenas e as variáveis são estatisticamente independentes. O valor analítico do projeto está em **mostrar o que os dados sustentam e o que não sustentam**, e não em forçar uma narrativa econômica que a base não contém.

## 10. Conclusão

**Conclusão executiva**

1. **Escala:** foram concedidos **~R$ 22,4 bi** entre 2015 e 2024, com **juros médios de ~48,7%** e **inadimplência média de ~10,6%**.
2. **Crescimento:** o crédito está **estagnado**, em torno de R$ 2,2 bi por ano (+2,5% em dez anos). As oscilações anuais (de −6,2% a +4,7%) não formam tendência.
3. **Geografia:** o **Sudeste concentra ~36%** do volume, mas por ter mais registros; o crédito médio por operação é parecido em todas as regiões.
4. **Risco:** **juros e inadimplência não se relacionam** (r ≈ −0,01), e a classificação de risco (Baixo, Médio, Alto) **distingue muito pouco** a inadimplência observada (~0,3 p.p. entre Baixo e Alto). Setores e modalidades diferem em menos de 1 p.p.
5. **Pontos de atenção:** as UFs com maior índice de criticidade merecem monitoramento, mas as diferenças são pequenas demais para decisões firmes.

**Limitações**

- Base **simulada**, com distribuições uniformes e variáveis independentes, o que explica a falta de padrões econômicos.
- Cobertura de **20 das 27 UFs**, com número desigual de registros por região, o que influencia os totais.
- Unidade do prazo (dias) **assumida**; sem informação sobre moeda constante, inflação ou política monetária no período.

**Próximos passos**

- Levar para o `app.py` (Streamlit): `resumo()`, os KPIs, os seis filtros obrigatórios (ano, mês, região, UF, modalidade, risco) e os gráficos das seções 8.1 a 8.6, com interpretação textual e conclusão executiva.
- Opcional: persistir a base em SQLite (`database/`) e consultar com SQLAlchemy.